# VSCS Colab template (P0-T5)

Template for every GPU stage: **install, mount Drive, check the GPU, call into `src/vscs`**.

> **This notebook contains no logic.** CLAUDE.md section 4.3: notebooks only install, mount and call into the package. If you find yourself writing an algorithm in a cell, it belongs in `src/vscs/` with a test.

Which stages run here (the laptop has no NVIDIA GPU):

| Stage | Task |
|---|---|
| Gaussian splatting | P1-T7 |
| Grounding DINO + SAM 2 masks | P2-T2 |
| Depth models | P4-T1 |
| Detection + tracking | P4-T4 |

**Runtime: T4.** Do not request an A100 without an ADR (CLAUDE.md section 1). Log approximate compute units used in today's devlog, and warn before any job over ~10 CU (R-11).

## 1. GPU check

Run this first. If there is no GPU, stop and change the runtime type.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# Colab ships its own torch build matched to the runtime. Never pip-install a
# different torch here: that is the most common way to break the runtime (R-12).
import torch

print(f"torch            {torch.__version__}")
print(f"cuda available   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device           {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"memory           {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total")
else:
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then rerun.")

## 2. Mount Drive

Everything durable goes to Drive: Colab disconnects and takes the local disk with it (R-11). Checkpoint **every stage**, not just the end of the job.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

# Single place the whole project lives on Drive. Keep this path stable: run
# folders under it are referenced from metrics/results.jsonl.
VSCS_DRIVE = Path("/content/drive/MyDrive/vscs")

CHECKPOINTS = VSCS_DRIVE / "checkpoints"   # model weights, downloaded once
RAW = VSCS_DRIVE / "data" / "raw"          # never written to by a notebook
PROCESSED = VSCS_DRIVE / "data" / "processed"

for d in (CHECKPOINTS, PROCESSED):
    d.mkdir(parents=True, exist_ok=True)

# Hello-world write, which is the P0-T5 acceptance check.
import datetime

stamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
probe = VSCS_DRIVE / "colab_hello.txt"
probe.write_text(f"VSCS Colab template OK at {stamp}\n")
print(f"wrote {probe}")
print(probe.read_text())

## 3. Get the code

Clone the repo rather than pasting code into cells, so the notebook runs the same package that the laptop tests.

In [ ]:
import os

REPO_URL = "https://github.com/AlvinKalaivannan/VSCS-Vehicle-Scanning-Collision-System-.git"
REPO_DIR = Path("/content/vscs")
BRANCH = "main"

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

os.chdir(REPO_DIR)
!git rev-parse --short HEAD

## 4. Install GPU-stage dependencies

Pinned in `envs/colab_requirements.txt`. Record any version combination that works, or any workaround you needed, in an ADR (R-12).

In [ ]:
!pip install --quiet -r envs/colab_requirements.txt

In [ ]:
import sys

# src layout: make the package importable without an editable install.
sys.path.insert(0, str(REPO_DIR / "src"))

import vscs
from vscs.common.config import load_config
from vscs.common.log import seed_everything, setup_logging

setup_logging()
print(f"vscs {vscs.__version__} imported from {Path(vscs.__file__).parent}")
print(f"risk horizon: {load_config('risk')['motion']['horizon_s']} s")

## 5. Call into the package

Replace this cell with the actual stage call. Two rules:

* seed every random process and log the seed (section 4.3);
* write outputs into a run folder on Drive, with the resolved config and the commit hash, so a metric can be traced back to what produced it.

In [ ]:
seed_everything(20260924)

# Example shape of a real stage call:
#
#   from vscs.seg.masks2d import run_masks
#   run_dir = run_masks(
#       recon_run=PROCESSED / "recon" / "20261019-1402_abc1234",
#       config=load_config("seg"),
#       out_root=PROCESSED,
#       checkpoints=CHECKPOINTS,
#   )
#   print(run_dir)

print("template only - nothing to run yet")

## 6. Before closing the tab

- [ ] Outputs are on **Drive**, not just `/content` (which disappears).
- [ ] Approximate compute units used noted for today's devlog (R-11).
- [ ] Any metric appended to `metrics/results.jsonl` **from a value computed in this run** - never a remembered number (sections 0.4, 7.4).
- [ ] Anything surprising written down in the devlog while you still remember it.